# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharmeenabdulwaheed/FlyRank-AI-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**My lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm choosing this lane because it matches a decision FlyRank editors already face every week: with
thousands of pages and limited editor hours, which ones get reviewed first? The starter dataset is
built for exactly this (`content_refresh_anonymized.csv`, one row per content item, with the
signals needed to judge whether a page is stale, declining, or still healthy). The starter pipeline
I already ran in Notebook 01 shows this lane is workable end-to-end: a hand-written baseline rule
and a random forest can both be scored on the same data with `precision@50`. I'm keeping Week 1-4
flexibility to switch to Lane 1 (Ranking Signal Analysis) if the refresh label turns out to be too
noisy once I dig into the warehouse release, but for now Lane 2 is my provisional pick.


In [1]:
# Find the repo root (same walk-up pattern as notebooks/01_first_look_and_discovery.ipynb)
# then confirm the grain: one row per content item, which is what a page-level
# refresh queue (Lane 2) needs.
import os
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", len(df), "| Unique content_id:", df["content_id"].nunique())
print("One row per content item:", len(df) == df["content_id"].nunique())


Rows: 30000 | Unique content_id: 30000
One row per content item: True


## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** Which content pages should an editor spend their limited review time
on first — refresh, expand, protect, or just monitor?

**Who acts on the output, and what do they do:** An SEO content editor at FlyRank (or a client-side
content owner) opens the top of a ranked queue and manually reviews those specific pages first,
instead of guessing or working alphabetically / by publish date.

**Cost of a wrong call:**
- **False positive** (we flag a healthy page as "refresh me"): wastes editor hours on a page that
  didn't need it — an opportunity cost, not a disaster.
- **False negative** (we miss a page that's actually declining and stays unreviewed): the page
  keeps losing visibility/traffic un-noticed until the next audit cycle — a slower, silent cost
  that compounds the longer it's missed.
- Because false negatives are quieter but more damaging over time, I'd rather the model
  over-flag than under-flag borderline pages — this will shape how I pick a threshold later.

**Why data/ML helps at all:** With tens of thousands of pages and only a handful of editor-hours a
week, a human can't manually re-check every page. A single hand-written rule (e.g. "flag anything
untouched for 90+ days") is a reasonable starting baseline, but it ignores how the signals interact
(traffic level, position, content type, freshness, engagement) — that's the gap a learned model or
a richer scoring approach can close, and it's exactly what the starter pipeline in Notebook 01
already demonstrated (a random forest beat the hand-written rule on precision@50).


In [2]:
# No new number needed here - the pipeline already run in Notebook 01 shows the
# baseline vs. model precision@50 gap that motivates "why ML helps" above.
import json

with open("outputs/model_results.json") as f:
    res = json.load(f)

base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Hand-written rule  precision@50: {base:.3f}")
print(f"Random forest       precision@50: {rf:.3f}")
print(f"-> a learned model already beats the simple rule by {rf/base:.1f}x on this small sample,")
print("   which is the evidence that a smarter scoring method is worth building here.")


Hand-written rule  precision@50: 0.240
Random forest       precision@50: 0.740
-> a learned model already beats the simple rule by 3.1x on this small sample,
   which is the evidence that a smarter scoring method is worth building here.


## 3. Quick look at the data (2-3 real numbers)

*Loaded below, straight from `data/raw/content_refresh_anonymized.csv` — nothing hardcoded.*


In [3]:
visible = df[df["impressions_90d"] >= 100]  # pages that actually get seen in search
declining_visible = visible[visible["trend_direction"] == "down"]

n_clients = df["client_id"].nunique()
share_declining = len(declining_visible) / len(visible)

stale_declining = declining_visible[
    declining_visible["freshness_tier"].isin(["91-180", "181+"])
]
share_stale_of_declining = len(stale_declining) / len(declining_visible)

print(f"1) Visible pages (impressions_90d >= 100): {len(visible):,} out of {len(df):,} total rows")
print(f"2) Of those visible pages, {share_declining:.1%} are declining (trend_direction == 'down')")
print(f"   -> {len(declining_visible):,} visible+declining pages: a real, sizeable review queue")
print(f"3) Of the visible+declining pages, {share_stale_of_declining:.1%} haven't been touched")
print(f"   in 91+ days (freshness_tier 91-180 or 181+) -> staleness and decline do line up,")
print(f"   which is exactly the signal a refresh-priority score should use")
print(f"\nAlso: {n_clients} distinct clients in the sample -> I'll need client-grouped")
print(f"train/test splits later so no client's pages leak across train and test.")


1) Visible pages (impressions_90d >= 100): 22,006 out of 30,000 total rows
2) Of those visible pages, 59.8% are declining (trend_direction == 'down')
   -> 13,152 visible+declining pages: a real, sizeable review queue
3) Of the visible+declining pages, 38.5% haven't been touched
   in 91+ days (freshness_tier 91-180 or 181+) -> staleness and decline do line up,
   which is exactly the signal a refresh-priority score should use

Also: 32 distinct clients in the sample -> I'll need client-grouped
train/test splits later so no client's pages leak across train and test.


## 4. Careful words: what I can and can't claim

**What I can claim:**
- **Observed** patterns in this anonymized sample (e.g. "visible + declining pages are more often
  stale than fresh in this dataset").
- **Directional** signals — a page scoring high on my model is *more likely, on this evidence,* to
  be worth reviewing than a page scoring low. Not a certainty for any single page.
- **Decision-support** — a ranked list an editor can use to prioritize limited review time, with
  reason codes explaining *why* a page was flagged, so a human still makes the final call.

**What I will never claim:**
- That I've reverse-engineered or "predicted" Google's ranking algorithm — I only have
  FlyRank-observed outcomes (impressions, clicks, sessions), never Google's internal signals.
- **Causal proof.** "Stale pages tend to decline together" is a correlation I can measure, not
  evidence that staleness *causes* decline (could easily be the reverse, or a shared third cause
  like seasonality or a competitor change).
- That refreshing a flagged page is *guaranteed* to fix it — the model estimates review priority,
  not the outcome of an edit.

**Leakage guardrail I'm already aware of (from `flyrank-data` skill):** `trend_direction` and
`trend_pct` define the label (`is_declining_label`) itself, so they can never be used as model
input features — only as the target, or for descriptive EDA like above.


In [4]:
# Sanity check: confirm the label-source columns are not accidentally sitting
# in a "features" list I might reuse later. (Nothing to compute yet - just a reminder pattern
# I'll reuse in the data-contract notebook.)
label_source_columns = ["trend_direction", "trend_pct"]
print("Never use as features (label leakage):", label_source_columns)
print("These define is_declining_label, so they may only be used as the target or in EDA.")


Never use as features (label leakage): ['trend_direction', 'trend_pct']
These define is_declining_label, so they may only be used as the target or in EDA.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.